In [2]:
import pandas as pd

df = pd.read_csv(
    "../data/raw/HI-Small_Trans.csv",
    parse_dates=["Timestamp"],
    nrows=500_000,
)

print(df.shape)


(500000, 11)


In [3]:
train_df = pd.DataFrame()

# Numerical
train_df["amount"] = df["Amount Paid"]

# Categorical
train_df["transaction_type"] = df["Payment Format"]
train_df["merchant_category"] = df["To Bank"].astype(str)
train_df["location"] = df["Receiving Currency"]
train_df["device_used"] = df["From Bank"].astype(str)
train_df["payment_channel"] = df["Payment Format"]

# Time features
train_df["hour"] = df["Timestamp"].dt.hour
train_df["day_of_week"] = df["Timestamp"].dt.dayofweek
train_df["month"] = df["Timestamp"].dt.month

train_df["is_weekend"] = (
    train_df["day_of_week"] >= 5
).astype(int)

# Target
train_df["target"] = df["Is Laundering"]

print(train_df.shape)
train_df.head()

(500000, 11)


,amount,transaction_type,merchant_category,location,device_used,payment_channel,hour,day_of_week,month,is_weekend,target
0,3697.34,Reinvestment,10,US Dollar,10,Reinvestment,0,3,9,0,0
1,0.01,Cheque,1,US Dollar,3208,Cheque,0,3,9,0,0
2,14675.57,Reinvestment,3209,US Dollar,3209,Reinvestment,0,3,9,0,0
3,2806.97,Reinvestment,12,US Dollar,12,Reinvestment,0,3,9,0,0
4,36682.97,Reinvestment,10,US Dollar,10,Reinvestment,0,3,9,0,0


In [4]:
from sklearn.model_selection import train_test_split

X = train_df.drop(columns=["target"])
y = train_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

print("\nTrain Labels")
print(y_train.value_counts())

print("\nTest Labels")
print(y_test.value_counts())

Train: (400000, 10)
Test : (100000, 10)

Train Labels
target
0    399846
1       154
Name: count, dtype: int64

Test Labels
target
0    99961
1       39
Name: count, dtype: int64


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_features = [
    "transaction_type",
    "merchant_category",
    "location",
    "device_used",
    "payment_channel",
]

numeric_features = [
    "amount",
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features,
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
            ),
            categorical_features,
        ),
    ]
)

print("Preprocessor Created Successfully")

Preprocessor Created Successfully


In [6]:
from xgboost import XGBClassifier

# Handle extreme class imbalance
scale_pos_weight = (
    y_train.value_counts()[0]
    /
    y_train.value_counts()[1]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            XGBClassifier(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42,
                scale_pos_weight=scale_pos_weight,
                tree_method="hist",
            ),
        ),
    ]
)

print("Training Started...")

model.fit(
    X_train,
    y_train,
)

print("Training Finished!")

Training Started...
Training Finished!


In [7]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

# Predict probabilities
y_prob = model.predict_proba(X_test)[:, 1]

# Predict labels
y_pred = model.predict(X_test)

print("=" * 60)
print("ROC-AUC")
print("=" * 60)
print(roc_auc_score(y_test, y_prob))

print("\n")

print("=" * 60)
print("Classification Report")
print("=" * 60)
print(classification_report(y_test, y_pred))

print("\n")

print("=" * 60)
print("Confusion Matrix")
print("=" * 60)
print(confusion_matrix(y_test, y_pred))

ROC-AUC
0.9890727896700227


Classification Report
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     99961
           1       0.03      0.82      0.05        39

    accuracy                           0.99    100000
   macro avg       0.51      0.90      0.52    100000
weighted avg       1.00      0.99      0.99    100000



Confusion Matrix
[[98774  1187]
 [    7    32]]


In [8]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

thresholds = np.arange(0.1, 1.0, 0.05)

print(
    f"{'Threshold':<10}"
    f"{'Precision':<12}"
    f"{'Recall':<12}"
    f"{'F1':<12}"
)

for threshold in thresholds:

    predictions = (
        y_prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_test,
        predictions,
        zero_division=0,
    )

    recall = recall_score(
        y_test,
        predictions,
        zero_division=0,
    )

    f1 = f1_score(
        y_test,
        predictions,
        zero_division=0,
    )

    print(
        f"{threshold:<10.2f}"
        f"{precision:<12.3f}"
        f"{recall:<12.3f}"
        f"{f1:<12.3f}"
    )

Threshold Precision   Recall      F1          
0.10      0.007       0.872       0.014       
0.15      0.008       0.846       0.016       
0.20      0.009       0.821       0.019       
0.25      0.011       0.821       0.021       
0.30      0.013       0.821       0.025       
0.35      0.015       0.821       0.030       
0.40      0.018       0.821       0.035       
0.45      0.021       0.821       0.041       
0.50      0.026       0.821       0.051       
0.55      0.032       0.821       0.062       
0.60      0.038       0.821       0.073       
0.65      0.044       0.821       0.084       
0.70      0.055       0.821       0.103       
0.75      0.070       0.821       0.129       
0.80      0.096       0.769       0.171       
0.85      0.158       0.769       0.262       
0.90      0.296       0.744       0.423       
0.95      0.744       0.744       0.744       


In [9]:
import joblib
import json
from pathlib import Path

MODEL_DIR = Path("../models/saved_models_v2")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save the complete pipeline (preprocessor + model)
joblib.dump(
    model,
    MODEL_DIR / "xgboost_v2.pkl",
)

metadata = {
    "model": "xgboost",
    "version": "v2",
    "threshold": 0.95,
    "roc_auc": 0.9891,
}

with open(
    MODEL_DIR / "xgboost_metadata_v2.json",
    "w",
) as f:
    json.dump(
        metadata,
        f,
        indent=4,
    )

print("Model Saved Successfully")

Model Saved Successfully
